# OpenPlaque — local aortic-root left-main search

This experiment is a fresh strategy in **source CCTA series 7**. It does not launch a long coronary graph search. Instead it searches only a small high-resolution aortic-root neighborhood, requires a short outward coronary-sized lumen immediately off the aortic wall, and only then follows that lumen locally.

The frozen RCA source-volume centerline is the positive calibration reference. **No LAD/LCX search is attempted here.** The left main must pass first.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls


In [ ]:
REUSE_SOURCE_CT = True
REUSE_ROOT_CROP = True
REUSE_RCA_CALIBRATION = True
REUSE_OSTIUM_CANDIDATES = True
REUSE_LEFT_MAIN = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install this branch and JPEG-lossless decoder


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-main-local-root-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas psutil "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.1"
import sys, os, gc, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label):
    p = psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Initialize workflow and inspect cache plan


In [ ]:
from openplaque.left_main_local_root import LeftMainLocalRootWorkflow, ALGORITHM_VERSION
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'root_crop': REUSE_ROOT_CROP,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'ostium_candidates': REUSE_OSTIUM_CANDIDATES,
    'left_main': REUSE_LEFT_MAIN,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LeftMainLocalRootWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
display(wf.cache_status())


## Step 5 — Load disk-backed source CCTA and build a small aortic-root crop

If the prior LowRAM series-7 `int16` cache exists, it is reused. The full CCTA remains disk-backed. Only a small root crop around the frozen RCA ostium is materialized for the local search.


In [ ]:
wf.load_source_ct()
wf.build_root_crop()
gc.collect(); ram('After source CT + root crop')
print('Root crop shape:', wf.root_ct.shape)
print('Root crop origin z,y,x:', wf.crop_lo)


## Step 6 — Calibrate strict lumen geometry from the accepted RCA

The RCA establishes the expected coronary radius, centering, circularity, attenuation, and core-to-ring contrast for this scan.


In [ ]:
rca = wf.calibrate_rca()
display(rca)
display(wf.rca_qc)


## Step 7 — High-resolution local left-ostium search

Candidate seeds are restricted to a 0.6–3.4 mm shell immediately outside the aorta, within ±10 mm of the RCA ostial level, and preferentially on the opposite aortic side. Short 10-mm direction bundles are tested before any extension. Large bright chambers should fail the coronary-sized cross-section gate here.


In [ ]:
ostia = wf.find_ostium_candidates()
display(ostia)
gc.collect(); ram('After local ostium search')


## Step 8 — Follow only rays that pass the ostium gate

A longer local tracker is launched only from a ray that already has coronary-sized serial cross-sections. The candidate is adaptively trimmed before a bifurcation or broad structure degrades the lumen geometry.


In [ ]:
left_main = wf.build_left_main()
print('Best left-main summary:')
display(wf.best_summary)
display(wf.best_qc)
gc.collect(); ram('After left-main tracking')


## Step 9 — Create decisive QC figures

The key figure is the direct side-by-side comparison of serial orthogonal sections from the frozen RCA and the proposed left main.


In [ ]:
figs = wf.plot_qc()
for f in figs:
    print('Saved:', f)
gc.collect(); ram('After figures')


## Step 10 — Package report-back ZIP


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_MAIN_LOCAL_ROOT_REPORT_BACK.zip')


## Step 11 — Report back

After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**. Do not proceed to LAD/LCX unless this left-main gate is visually convincing.
